In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
df_couple = pd.read_pickle(r"C:\Users\asus\Desktop\acs_ssc_lasso_input.pkl")
n_couples = len(df_couple)

# Age filter
df_couple = df_couple[
    df_couple['age'].between(18,60) & 
    df_couple['sp_age'].between(18,60)
].copy()
n_couples = len(df_couple)
print(f"Sample size: {n_couples:,}")

Sample size: 37,907


In [2]:
# Restrict the sample before running LASSO
df_couple = df_couple[df_couple['age'].between(18, 60)]
df_couple = df_couple[df_couple['sp_age'].between(18, 60)]

# Handle extreme earnings (top-coding / winsorization)
# Observed earnings are higher than in the paper (e.g., 592 vs 442),
# so this step helps reduce the influence of outliers
upper_limit = df_couple['r_incearn'].quantile(0.99)

df_couple['r_incearn'] = df_couple['r_incearn'].clip(upper=upper_limit)
df_couple['sp_incearn'] = df_couple['sp_incearn'].clip(upper=upper_limit)

# Recompute the share of individuals with positive earnings
# This is a key input for calibrating the LASSO participation model
# The high participation rate in the paper (~96%) is partly driven
# by sample restrictions that exclude non-working subpopulations
mean_pos_all = df_couple['r_posincearn'].mean()

print(f"Share of positive earnings in the sample: {mean_pos_all:.3%}")

Share of positive earnings in the sample: 90.500%


In [3]:
# 2. [Core Step] Construct pooled individual-level dataset

# Extract characteristics and observed earnings for household heads (r)
df_r = df_couple[
    ['year', 'r_incearn', 'r_agegroup', 'r_edugroup', 'r_race',
     'r_occbroad', 'r_degbroad', 'c_children', 'statefip']
].rename(columns={
    'r_incearn': 'incearn',
    'r_agegroup': 'agegroup',
    'r_edugroup': 'edugroup',
    'r_race': 'race',
    'r_occbroad': 'occbroad',
    'r_degbroad': 'degbroad'
})

# Extract characteristics and observed earnings for spouses (sp)
df_sp = df_couple[
    ['year', 'sp_incearn', 'sp_agegroup', 'sp_edugroup', 'sp_race',
     'sp_occbroad', 'sp_degbroad', 'c_children', 'statefip']
].rename(columns={
    'sp_incearn': 'incearn',
    'sp_agegroup': 'agegroup',
    'sp_edugroup': 'edugroup',
    'sp_race': 'race',
    'sp_occbroad': 'occbroad',
    'sp_degbroad': 'degbroad'
})

# Stack the two datasets to form a pooled sample of all individuals
# (This ensures that heads and spouses share the same model parameters)
df_indiv = pd.concat([df_r, df_sp], axis=0, ignore_index=True)

In [5]:
# 3. Prepare features

# Define categorical and numerical variables
cat_cols = ['agegroup', 'edugroup', 'race', 'occbroad', 'degbroad', 'statefip']
num_cols = ['c_children']

# One-hot encode categorical variables (drop first category to avoid multicollinearity)
X_df = pd.get_dummies(
    df_indiv[cat_cols + num_cols],
    columns=cat_cols,
    drop_first=True,
    dtype=float
)

# Construct pairwise interaction terms (only based on individual-level characteristics)
# Keep interactions with sufficient support to avoid extremely sparse features
base_cols = list(X_df.columns)
interacts = []

for i in range(len(base_cols)):
    for j in range(i + 1, len(base_cols)):
        c1, c2 = base_cols[i], base_cols[j]
        s = X_df[c1] * X_df[c2]
        
        # Keep interaction if it appears frequently enough in the data
        if s.sum() >= 10:
            interacts.append(s.rename(f"{c1}_x_{c2}"))

# Append interaction terms to the design matrix
if interacts:
    X_df = pd.concat([X_df, pd.concat(interacts, axis=1)], axis=1)

# Standardize features (important for Lasso)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_df)

# Outcome variable (individual earnings)
y = df_indiv['incearn'].values

# Training sample: year 2012
train_mask = df_indiv['year'] == 2012

In [6]:
# 4. Train a single global LASSO model (pooled individual-level model)

print("--- Training pooled individual earnings model ---")

# Step 1: Predict probability of having positive earnings (extensive margin)
y1 = (y > 0).astype(float)

m1 = LassoCV(
    cv=10,
    max_iter=5000,
    n_jobs=-1,
    random_state=42
)
m1.fit(X_sc[train_mask], y1[train_mask])

# Step 2: Predict earnings level conditional on working (intensive margin)
pos_train = train_mask & (y > 0)

m2 = LassoCV(
    cv=10,
    max_iter=5000,
    n_jobs=-1,
    random_state=42
)
m2.fit(X_sc[pos_train], y[pos_train])

--- Training pooled individual earnings model ---


,eps,0.001
,n_alphas,'deprecated'
,alphas,'warn'
,fit_intercept,True
,precompute,'auto'
,max_iter,5000
,tol,0.0001
,copy_X,True
,cv,10
,verbose,False
,n_jobs,-1


In [8]:
# --- Make sure all sample cleaning (e.g., drops/restrictions) has been completed before running this cell ---
# Split standardized features into head and spouse
X_r_sc = X_sc[:n_couples]
X_sp_sc = X_sc[n_couples:]

# 5.1 Compute global threshold for participation (extensive margin)
hat_pos_prob_all = m1.predict(X_sc)

# Compute the actual share of individuals with positive earnings in the cleaned sample
mean_pos_actual = (df_indiv['incearn'] > 0).mean()
print(f"Actual share of positive earnings after cleaning: {mean_pos_actual:.3%}")

# Align predicted participation with observed share using a quantile threshold
# Note: if mean_pos_actual is still far below ~96%, additional sample restrictions may be needed
threshold = np.quantile(hat_pos_prob_all, 1 - mean_pos_actual)

# 5.2 Predictions for household heads (first n observations)
hat_pos_r_prob = hat_pos_prob_all[:n_couples]
hat_pos_r = (hat_pos_r_prob >= threshold).astype(float)

# Predicted earnings conditional on working (truncate at zero)
hat_earn_r = np.maximum(m2.predict(X_r_sc), 0)

df_couple["hat_incearn_r"] = np.where(hat_pos_r == 1, hat_earn_r, 0.0)

# 5.3 Predictions for spouses (last n observations)
hat_pos_sp_prob = hat_pos_prob_all[n_couples:]
hat_pos_sp = (hat_pos_sp_prob >= threshold).astype(float)

hat_earn_sp = np.maximum(m2.predict(X_sp_sc), 0)

df_couple["hat_incearn_sp"] = np.where(hat_pos_sp == 1, hat_earn_sp, 0.0)

df_couple["hat_pos_r"] = hat_pos_r
df_couple["hat_pos_sp"] = hat_pos_sp

Actual share of positive earnings after cleaning: 88.383%


In [9]:
# 6. Compute couple-level outcomes

# Total predicted earnings at the couple level
df_couple["hat_c_incearn"] = (
    df_couple["hat_incearn_r"] + df_couple["hat_incearn_sp"]
)

# Earnings share of the primary earner within the couple
# (i.e., max share between the two partners)
df_couple["hat_c_split"] = np.where(
    df_couple["hat_c_incearn"] > 0,
    np.maximum(
        df_couple["hat_incearn_r"],
        df_couple["hat_incearn_sp"]
    ) / df_couple["hat_c_incearn"],
    0.5  # Assign 0.5 when total earnings are zero
)

In [10]:
# ── Diagnostics vs Table 2 ───────────────────────────────────────
print("="*60)
print("DIAGNOSTICS vs. Table 2")
print("="*60)

print("\n--- Share positive earnings ---")
print(f"  Respondent  reported={(df_couple['r_incearn']>0).mean():.3f}  "
      f"predicted={df_couple['hat_pos_r'].mean():.3f}  "
      f"(paper predicted: 0.963 mar / 0.969 coh)")
print(f"  Spouse      reported={(df_couple['sp_incearn']>0).mean():.3f}  "
      f"predicted={df_couple['hat_pos_sp'].mean():.3f}")

print("\n--- Couple predicted total earnings ---")
for label, mask in [("Married", df_couple["sscouple_mar"]),
                    ("Cohabiting", df_couple["sscouple_coh"])]:
    sub = df_couple[mask]
    print(f"  {label}: {sub['hat_c_incearn'].mean():>10,.0f}  "
          f"(paper: mar≈110,729  coh≈102,953)")

print("\n--- Couple predicted earnings split ---")
for label, mask in [("Married", df_couple["sscouple_mar"]),
                    ("Cohabiting", df_couple["sscouple_coh"])]:
    sub = df_couple[mask]
    pos = sub["hat_c_incearn"] > 0
    print(f"  {label}: {sub.loc[pos,'hat_c_split'].mean():.3f}  "
          f"(paper: mar≈0.648  coh≈0.641)")

print("\n--- Correlation reported vs predicted (positive earners) ---")
for pfx, label in [("r", "Respondent"), ("sp", "Spouse")]:
    both = (df_couple[f"{pfx}_incearn"] > 0) & (df_couple[f"hat_incearn_{pfx}"] > 0)
    r = np.corrcoef(df_couple.loc[both, f"{pfx}_incearn"],
                    df_couple.loc[both, f"hat_incearn_{pfx}"])[0, 1]
    print(f"  {label}: r={r:.3f}")

DIAGNOSTICS vs. Table 2

--- Share positive earnings ---
  Respondent  reported=0.905  predicted=0.893  (paper predicted: 0.963 mar / 0.969 coh)
  Spouse      reported=0.863  predicted=0.877

--- Couple predicted total earnings ---
  Married:    111,938  (paper: mar≈110,729  coh≈102,953)
  Cohabiting:    101,670  (paper: mar≈110,729  coh≈102,953)

--- Couple predicted earnings split ---
  Married: 0.626  (paper: mar≈0.648  coh≈0.641)
  Cohabiting: 0.628  (paper: mar≈0.648  coh≈0.641)

--- Correlation reported vs predicted (positive earners) ---
  Respondent: r=0.412
  Spouse: r=0.415


In [11]:
# Save
OUT_PATH = r"C:\Users\asus\Desktop\acs_ssc_predicted_fixed.pkl"
df_couple.to_pickle(OUT_PATH)
print(f"{OUT_PATH}")

C:\Users\asus\Desktop\acs_ssc_predicted_fixed.pkl
